# ViralTweets — Parquet Data Analysis

This notebook analyses `final_dataset_since_october_2022.parquet.gzip` (available in both
`classification/model_with_extra_features/` and `classification/model_with_only_language_models/`)
and the `all_metric_stats.csv` file, then documents what further work can be done.

Paper: *Measuring and Detecting Virality on Social Media: The Case of Twitter's Viral Tweets Topic*

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = 'classification/model_with_extra_features/final_dataset_since_october_2022.parquet.gzip'
PARQUET_PATH_LM = 'classification/model_with_only_language_models/final_dataset_since_october_2022.parquet.gzip'
METRICS_CSV = 'all_metric_stats.csv'

df = pd.read_parquet(PARQUET_PATH)
df_lm = pd.read_parquet(PARQUET_PATH_LM)
metrics = pd.read_csv(METRICS_CSV)

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

## 1. Dataset overview

In [ ]:
print('Shape:', df.shape)
print()
print('Columns and dtypes:')
print(df.dtypes)

In [ ]:
viral_count = df['viral'].sum()
total = len(df)

print(f'Total tweets  : {total:>10,}')
print(f'Viral tweets  : {viral_count:>10,}  ({viral_count/total*100:.2f}%)')
print(f'Non-viral     : {total-viral_count:>10,}  ({(total-viral_count)/total*100:.2f}%)')
print()
print(f'Date range    : {df["created_at"].min()} → {df["created_at"].max()}')
print(f'Unique authors: {df["author_id"].nunique():,}')
print(f'Language(s)   : {df["lang"].unique()}')

## 2. Engagement metrics — viral vs non-viral

In [ ]:
engagement_cols = ['retweet_count', 'reply_count', 'like_count', 'quote_count']
summary = df.groupby('viral')[engagement_cols].agg(['median', 'mean', 'max'])
summary.index = ['Non-viral', 'Viral']
print(summary.to_string())

**Key finding:** Viral tweets have median retweet counts ~10,600× higher than non-viral (10,631 vs 0)
and median like counts ~20,886× higher (83,546 vs 4). Engagement is the strongest raw signal.

## 3. Author / content features — viral vs non-viral

In [ ]:
feature_cols = ['tweet_length', 'nb_of_hashtags', 'nb_of_mentions',
                'followers_count', 'following_count', 'tweet_count']

print('--- Median values ---')
print(df.groupby('viral')[feature_cols].median().rename(index={False:'Non-viral', True:'Viral'}).T.to_string())

print()
print('--- Boolean features (% True) ---')
bool_cols = ['possibly_sensitive', 'has_media', 'protected', 'verified']
for col in bool_cols:
    nv = df[~df['viral']][col].mean()*100
    v  = df[df['viral']][col].mean()*100
    print(f'  {col:<22}: Non-viral {nv:.1f}%   Viral {v:.1f}%')

**Key findings:**
- Viral tweets are **longer** (median 72 vs 50 chars) and use **fewer mentions** (0 vs 1).
- Viral tweets are **far more likely to contain media** (62% vs 19%).
- `verified` and `followers_count` barely differ, suggesting raw account status is not a strong predictor.

## 4. Sentiment distribution

In [ ]:
sentiment_table = df.groupby(['viral', 'sentiment']).size().unstack(fill_value=0)
sentiment_table.index = ['Non-viral', 'Viral']
sentiment_pct = sentiment_table.div(sentiment_table.sum(axis=1), axis=0) * 100
print('Counts:')
print(sentiment_table)
print()
print('Percentages:')
print(sentiment_pct.round(1))

**Key finding:** Both classes skew heavily negative (~60-74%), but viral tweets are **even more negative** (74% vs 61%).
Negative emotional content correlates with higher virality.

## 5. Virality detection metrics (AUC-ROC comparison)

In [ ]:
auc_summary = metrics.groupby('metric_name')['roc1'].first().sort_values(ascending=False)
print('Metric                  AUC-ROC')
print('-' * 35)
for name, auc in auc_summary.items():
    if name != 'unused':
        print(f'  {name:<22} {auc:.4f}')

**Key finding:**
- **Influence Score** (AUC 0.961) is the best single metric for detecting virality.
- **RT > Avg. RT** (0.927) and **RT / Followers** (0.917) also perform strongly.
- Simple thresholding metrics like **RT Percentile** (0.841) and **RT > T** (0.827) are weakest.

## 6. Classification model results

In [ ]:
results = {
    'Model': ['BERTweet', 'BERT-tiny', 'BERT-base-cased', 'RoBERTa-base'],
    'Extra features — Balanced Acc.': [0.777, 0.723, 0.704, 0.729],
    'Extra features — F1': [0.793, 0.762, 0.734, 0.761],
    'LM only — Balanced Acc.': [0.748, 0.729, 0.694, 0.742],
    'LM only — F1': [0.766, 0.755, 0.714, 0.764],
}
results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

**Key findings:**
- **BERTweet** (trained on tweets) is the best model in both settings (balanced acc. 0.777 with features).
- Adding extra features (verified, tweet_length, has_media, sentiment, nb_of_hashtags, nb_of_mentions) consistently improves performance over language-model-only classifiers.
- The gain from extra features is +2-3 pp balanced accuracy across all models.

## 7. Difference between the two parquet files

In [ ]:
diff_cols = [c for c in df.columns if not df[c].equals(df_lm[c])]
print('Columns that differ between datasets:', diff_cols)
print()
# Inspect how they differ
for col in diff_cols:
    print(f'Column: {col}')
    print('  model_with_extra_features sample:', df[col].head(3).tolist())
    print('  model_with_only_lm sample:       ', df_lm[col].head(3).tolist())

## 8. What can be done — opportunities

### 8.1 Improved classification

| Idea | Details |
|---|---|
| **Larger / newer LMs** | GPT-based tweet encoders, LLaMA fine-tune, or Llama-3-based tweet classifiers could push accuracy above 0.80. |
| **Temporal features** | Add hour-of-day, day-of-week, time since account creation — patterns in the data suggest timing matters. |
| **Graph features** | Follower-graph centrality (PageRank) of the author; virality often spreads from hub nodes. |
| **Better class balancing** | Current dataset is 0.2% viral. SMOTE, focal loss, or cost-sensitive training could help further. |
| **Multi-modal** | 62% of viral tweets contain media — include image/video embeddings as additional features. |
| **Ensemble** | Combine the best LM with metric-based features (Influence Score has AUC 0.961) for a hybrid model. |

### 8.2 Metric analysis

| Idea | Details |
|---|---|
| **Combine top metrics** | Influence Score + RT > Avg. RT could be fused into a composite score. |
| **Calibration** | Calibrate probability estimates to give real-world P(viral) for production use. |
| **User-level virality profile** | `retweet_count_user_viral_threshold` varies widely; personalise the threshold per author. |

### 8.3 Data / research extensions

| Idea | Details |
|---|---|
| **Extend time window** | Dataset covers only Oct–Nov 2022. A longer window would reveal seasonal/event-driven virality. |
| **Multi-language** | All tweets are English (`lang=en`). Extending to other languages would test generalisation. |
| **Causal analysis** | Does media *cause* virality, or do viral accounts simply post more media? A causal graph would clarify. |
| **Early virality detection** | Use tweet features available at t=0 (before engagement is observed) to predict future virality — a real-time use case. |
| **Topic analysis** | `topic_domains` and `topic_entities` fields are currently sparse; enriching them (e.g. with an LLM) could unlock topic-level virality trends. |

### 8.4 Engineering / reproducibility

| Idea | Details |
|---|---|
| **Streamlined pipeline** | Wrap preprocessing + training into a single CLI or Makefile for one-command reproducibility. |
| **Experiment tracking** | Add MLflow / W&B logging to the existing `classification.py` scripts. |
| **Model export** | Serialize the best checkpoint (BERTweet + extra features) to ONNX or HuggingFace Hub for inference. |